In [81]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [ ]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0.7,
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

In [ ]:
from langchain.tools import tool
import json
import requests


@tool
def get_fun_fact():
    """
    Returns today's fun fact from Useless Facts API
    """
    url = "https://uselessfacts.jsph.pl/api/v2/facts/today"
    response = requests.get(url)
    resp_dict = json.loads(response.text)
    resp_text = resp_dict.get("text")
    fact = f"Did you know: {resp_text}"
    return fact

tools = [get_fun_fact]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

In [ ]:
from langchain_core.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator


class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

In [ ]:
from langchain_core.messages import SystemMessage


def llm_call(state: dict):
    """LLM decides whether to call a tool or not"""
    return {
        "messages": [
            model_with_tools.invoke(
                [
                    SystemMessage(
                        content="You are a helpful assistant tasked with stating interesting and fun facts about cats and dogs."
                    )
                ]
                + state["messages"]
            )
        ],
        "llm_calls": state.get('llm_calls', 0) + 1
    }

In [ ]:
from langchain_core.messages import ToolMessage


def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END


def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"

    # Otherwise, we stop (reply to the user)
    return END

In [ ]:
#Build workflow
agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# Add edges to connect nodes
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)
agent_builder.add_edge("tool_node", "llm_call")

# Compile the agent
agent = agent_builder.compile()

# Show the agent
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))



In [ ]:
# Invoke
from langchain_core.messages import HumanMessage
messages = [HumanMessage(content="Give me the fun fact for today")]
messages = agent.invoke({"messages": messages})
for m in messages["messages"]:
    m.pretty_print()

In [78]:
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path="movie_info.csv", source_column="title")

movie_ratings = loader.load()

movie_ratings_test = movie_ratings[0:9]

In [61]:
movie_ratings[1].page_content.replace("\n", " ")

'title: Airport url: https://www.rottentomatoes.com/m/airport release_date: Released Apr 5, 1970 critic_score: 75% audience_score: 54%'

In [45]:
from openai import OpenAI
import pandas as pd

movie_df = pd.read_csv("movie_info.csv").dropna().drop_duplicates(subset='title')


In [55]:
client = OpenAI(default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1')


In [63]:
def get_embedding(text, model="text-embedding-3-small"):
    text = text.page_content.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

In [65]:
embeddings = [get_embedding(doc) for doc in movie_ratings_test]
embeddings

[[-0.03497118502855301,
  0.010283811949193478,
  -0.06218022108078003,
  -0.052591681480407715,
  -0.024593979120254517,
  -0.0208996944129467,
  -0.019644051790237427,
  0.013625272549688816,
  -0.007367816753685474,
  -0.028163738548755646,
  0.0003554193244781345,
  -0.02235250361263752,
  -0.017900681123137474,
  -0.01161209400743246,
  -0.007590926717966795,
  -0.011373418383300304,
  0.014071492478251457,
  0.005894253496080637,
  0.02426190860569477,
  -0.007829602807760239,
  0.048648346215486526,
  0.039557911455631256,
  -0.008135730400681496,
  -0.02363927662372589,
  0.06271983683109283,
  0.03281272575259209,
  -0.014818650670349598,
  0.007274421863257885,
  0.017319558188319206,
  -0.00949514377862215,
  -0.022269485518336296,
  -0.02743733488023281,
  0.05931611359119415,
  -0.023556258529424667,
  -0.006656977813690901,
  -0.023909084498882294,
  0.03712964430451393,
  -0.05711614340543747,
  0.008031957782804966,
  -0.03750322386622429,
  0.0015929011860862374,
  -0.

In [84]:
import chromadb

chroma_client = chromadb.PersistentClient(path= "chromadb")


In [85]:
collection = chroma_client.get_or_create_collection("movie_ratings")

ids = [f"id{i}" for i in range(len(movie_ratings_test))]
movie_list = [doc.page_content for doc in movie_ratings_test]

collection.add(embeddings= embeddings,
               documents = movie_list,
               ids = ids)

In [86]:
def query_chromadb(query, top_n = 2):
    query_embedding = get_embedding(query)
    results = collection.query(query_embeddings = [query_embedding], n_results = top_n)
    return [(id, score, text) for id, score, text in zip(results['ids'][0], results['distances'][0], results['documents'][0])]

In [88]:
query = "What is the highest rated movie?"
query_chromadb(query, top_n=3)

AttributeError: 'str' object has no attribute 'page_content'